In [35]:
from pathlib import Path
import atoti as tt
from atoti_jdbc import JdbcLoad

In [36]:
# Tạo session
session = tt.Session.start(
    tt.SessionConfig(
        user_content_storage=Path("./atoti_content")
    )
)

# Kiểm tra URL app Atoti
print(session.url)

http://localhost:56408


In [37]:
'''
import atoti as tt
from pathlib import Path

with tt.Session.start(
    tt.SessionConfig(user_content_storage=Path("./atoti_content"))
) as session:
    cube = session.create_cube(...)
    # code xử lý ở đây
    ...
# Khi thoát khỏi khối with, session.close() được gọi tự động
'''

'\nimport atoti as tt\nfrom pathlib import Path\n\nwith tt.Session.start(\n    tt.SessionConfig(user_content_storage=Path("./atoti_content"))\n) as session:\n    cube = session.create_cube(...)\n    # code xử lý ở đây\n    ...\n# Khi thoát khỏi khối with, session.close() được gọi tự động\n'

In [38]:
# ===============================================
# 1️⃣ Thông tin kết nối PostgreSQL
# ===============================================
POSTGRES = {
    "host": "localhost",
    "port": 5433,
    "db": "cars",
    "user": "admin",
    "password": "admin123",
}

# Chuỗi kết nối JDBC — phải có prefix "jdbc:postgresql://"
postgres_url = (
    f"jdbc:postgresql://{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['db']}?"
    f"user={POSTGRES['user']}&password={POSTGRES['password']}"
)


In [39]:
from sqlalchemy import create_engine, MetaData

# 1️⃣ Kết nối PostgreSQL
url = (
    f"postgresql+psycopg2://{POSTGRES['user']}:{POSTGRES['password']}"
    f"@{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['db']}"
)
engine = create_engine(url)

# 2️⃣ Đọc metadata của schema
metadata = MetaData()
metadata.reflect(engine, schema="marts")

# --- Tạo dict bảng với primary key do mình tự gán ---
# Ví dụ: tables_pk = {"fact_sales": "sales_id", "dim_customer": "customer_id"}
tables_pk = {
    "dim_car_details": "car_details_id",
    "dim_car_general": "car_general_id",
    "dim_car_specs": "car_specs_id",
    "dim_date": "date_id",
    "fact_car_listing": "id"
}

# --- Load dữ liệu vào Atoti ---
tables = {}
for table_fullname, table in metadata.tables.items():
    clean_name = table_fullname.split(".")[-1]  # chỉ tên bảng

    # Nếu bảng không có trong dict tables_pk thì bỏ qua
    if clean_name not in tables_pk:
        print(f"⚠️ Skipping table {clean_name}, no primary key assigned")
        continue

    pk = tables_pk[clean_name]

    query = f"SELECT * FROM {table_fullname}"  # vẫn giữ schema trong query
    jdbc_load = JdbcLoad(query=query, url=postgres_url, driver="org.postgresql.Driver")

    data_types = session.tables.infer_data_types(jdbc_load)
    print(data_types)

    # Tạo table trong Atoti
    tables[clean_name] = session.create_table(
        clean_name,
        data_types=data_types,
        keys={pk} if pk else set(),
        default_values={
            # Default values cho scalar types
            **{col_name: 0 for col_name in data_types 
               if data_types[col_name] in ["int", "long"]},
            **{col_name: 0.0 for col_name in data_types 
               if data_types[col_name] in ["float", "double"]},
            # Default values cho array types
            **{col_name: [0] for col_name in data_types 
               if data_types[col_name] in ["int[]", "long[]"]},
            **{col_name: [0.0] for col_name in data_types 
               if data_types[col_name] in ["float[]", "double[]"]},
        }
    )
    tables[clean_name].load(jdbc_load)
    print(f"✅ Loaded table: {clean_name} | PK: {pk}")
    print(tables[clean_name].head(3).sort_index())

{'car_details_id': 'long', 'year': 'long', 'origin': 'String', 'mileage_category': 'String'}
✅ Loaded table: dim_car_details | PK: car_details_id
                year     origin   mileage_category
car_details_id                                    
3               1989  nhập khẩu  50.001-100.000 km
7               1992  nhập khẩu  50.001-100.000 km
9               1994  nhập khẩu       > 100.000 km
{'id': 'long', 'price': 'long', 'mileage': 'long', 'car_general_id': 'long', 'car_details_id': 'long', 'car_specs_id': 'long', 'date_id': 'LocalDate'}
✅ Loaded table: fact_car_listing | PK: id
           price  mileage  car_general_id  car_details_id  car_specs_id  \
id                                                                        
13992  460000000    76000             260             215             2   
19327  238000000   160000             327             123             2   
29442  355000000    50000             434              85            26   

         date_id  
id         

In [40]:
dim_tables = {
    "dim_car_details": "car_details_id",
    "dim_car_general": "car_general_id",
    "dim_car_specs": "car_specs_id",
    "dim_date": "date_id"
}

for dim_name, key in dim_tables.items():
    dim_table = session.tables[dim_name]
    tables['fact_car_listing'].join(dim_table, tables['fact_car_listing'][key] == dim_table[key])

In [41]:
session.tables.schema

```mermaid
erDiagram
  "dim_car_details" {
    non-null long PK "car_details_id"
    non-null long "year"
    non-null String "origin"
    non-null String "mileage_category"
  }
  "fact_car_listing" {
    non-null long PK "id"
    non-null long "price"
    non-null long "mileage"
    non-null long "car_general_id"
    non-null long "car_details_id"
    non-null long "car_specs_id"
    non-null LocalDate "date_id"
  }
  "dim_car_general" {
    non-null long PK "car_general_id"
    non-null String "brand"
    non-null String "model"
    non-null String "body_style"
  }
  "dim_car_specs" {
    non-null long PK "car_specs_id"
    non-null String "transmission"
    non-null String "engine"
    non-null String "drivetrain"
    non-null String "exterior_color"
    non-null String "interior_color"
  }
  "dim_date" {
    non-null LocalDate PK "date_id"
    non-null long "day_value"
    non-null long "month_value"
    non-null long "year_value"
  }
  "fact_car_listing" }o--o| "dim_car_details" : "car_details_id == car_details_id"
  "fact_car_listing" }o--o| "dim_car_general" : "car_general_id == car_general_id"
  "fact_car_listing" }o--o| "dim_car_specs" : "car_specs_id == car_specs_id"
  "fact_car_listing" }o--o| "dim_date" : "date_id == date_id"
```


In [42]:
# ===============================================
# 7️⃣ Tạo cube và mở UI
# ===============================================
cube = session.create_cube(tables["fact_car_listing"], mode='no_measures')

In [43]:
# Aliasing the hierarchies property to a shorter variable name because we will use it a lot.
h = cube.hierarchies
h

{('dim_car_specs', 'exterior_color'): <atoti.hierarchy.Hierarchy object at 0x11fdec5e0>, ('dim_car_specs', 'drivetrain'): <atoti.hierarchy.Hierarchy object at 0x11fdee290>, ('dim_car_specs', 'engine'): <atoti.hierarchy.Hierarchy object at 0x11fded4b0>, ('dim_car_specs', 'transmission'): <atoti.hierarchy.Hierarchy object at 0x11fdefb50>, ('dim_car_specs', 'interior_color'): <atoti.hierarchy.Hierarchy object at 0x11fdec340>, ('dim_car_details', 'origin'): <atoti.hierarchy.Hierarchy object at 0x11fdef6a0>, ('dim_car_details', 'mileage_category'): <atoti.hierarchy.Hierarchy object at 0x11fdef9d0>, ('dim_car_general', 'brand'): <atoti.hierarchy.Hierarchy object at 0x11fdefc10>, ('dim_car_general', 'body_style'): <atoti.hierarchy.Hierarchy object at 0x11fdefa30>, ('dim_car_general', 'model'): <atoti.hierarchy.Hierarchy object at 0x11fdede40>, ('fact_car_listing', 'date_id'): <atoti.hierarchy.Hierarchy object at 0x11fdef940>, ('fact_car_listing', 'id'): <atoti.hierarchy.Hierarchy object at 0x11fdec100>}

In [44]:
h.update(
    {
        ('dim_car_details', 'year'): [tables['dim_car_details']['year']],
        ('dim_car_general', 'name'): [
            tables['dim_car_general']['brand'],
            tables['dim_car_general']['model']
        ],
        ('dim_date', 'date'): [
            tables['dim_date']['year_value'],
            tables['dim_date']['month_value'],
            tables['dim_date']['day_value']
        ]
    }
)

In [45]:
h

{('dim_car_specs', 'exterior_color'): <atoti.hierarchy.Hierarchy object at 0x11c241120>, ('dim_car_specs', 'drivetrain'): <atoti.hierarchy.Hierarchy object at 0x11c243070>, ('dim_car_specs', 'engine'): <atoti.hierarchy.Hierarchy object at 0x11c243df0>, ('dim_car_specs', 'transmission'): <atoti.hierarchy.Hierarchy object at 0x11c242020>, ('dim_car_specs', 'interior_color'): <atoti.hierarchy.Hierarchy object at 0x11c2401f0>, ('dim_date', 'date'): <atoti.hierarchy.Hierarchy object at 0x11c242d10>, ('dim_car_details', 'origin'): <atoti.hierarchy.Hierarchy object at 0x11fdc8fa0>, ('dim_car_details', 'mileage_category'): <atoti.hierarchy.Hierarchy object at 0x11fdca380>, ('dim_car_details', 'year'): <atoti.hierarchy.Hierarchy object at 0x11fdcb610>, ('dim_car_general', 'brand'): <atoti.hierarchy.Hierarchy object at 0x11fdcaf20>, ('dim_car_general', 'body_style'): <atoti.hierarchy.Hierarchy object at 0x11fdca200>, ('dim_car_general', 'model'): <atoti.hierarchy.Hierarchy object at 0x11fdc9a20>, ('dim_car_general', 'name'): <atoti.hierarchy.Hierarchy object at 0x11fdc9270>, ('fact_car_listing', 'date_id'): <atoti.hierarchy.Hierarchy object at 0x11fdcb280>, ('fact_car_listing', 'id'): <atoti.hierarchy.Hierarchy object at 0x11fdcb0d0>}

In [46]:
del h['fact_car_listing', 'date_id']
del h['fact_car_listing', 'id']
del h['dim_car_general', 'brand']
del h['dim_car_general', 'model']

h

{('dim_car_specs', 'exterior_color'): <atoti.hierarchy.Hierarchy object at 0x11fdef550>, ('dim_car_specs', 'drivetrain'): <atoti.hierarchy.Hierarchy object at 0x11c2417b0>, ('dim_car_specs', 'engine'): <atoti.hierarchy.Hierarchy object at 0x11c241db0>, ('dim_car_specs', 'transmission'): <atoti.hierarchy.Hierarchy object at 0x11c242da0>, ('dim_car_specs', 'interior_color'): <atoti.hierarchy.Hierarchy object at 0x11c240fd0>, ('dim_date', 'date'): <atoti.hierarchy.Hierarchy object at 0x11c242d40>, ('dim_car_details', 'origin'): <atoti.hierarchy.Hierarchy object at 0x11c241cf0>, ('dim_car_details', 'mileage_category'): <atoti.hierarchy.Hierarchy object at 0x11c242560>, ('dim_car_details', 'year'): <atoti.hierarchy.Hierarchy object at 0x11c243d60>, ('dim_car_general', 'body_style'): <atoti.hierarchy.Hierarchy object at 0x11c243430>, ('dim_car_general', 'name'): <atoti.hierarchy.Hierarchy object at 0x11c2403d0>}

In [47]:
m = cube.measures
m

{'contributors.COUNT': <atoti.measure.Measure object at 0x120446350>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x120445180>}

In [48]:
m["price.SUM"] = tt.agg.sum(tables['fact_car_listing']["price"])
m["price.MEAN"] = tt.agg.mean(tables['fact_car_listing']["price"])

m["mileage.SUM"] = tt.agg.sum(tables['fact_car_listing']["mileage"])
m["mileage.MEAN"] = tt.agg.mean(tables['fact_car_listing']["mileage"])

In [49]:
m

{'contributors.COUNT': <atoti.measure.Measure object at 0x120445c90>, 'update.TIMESTAMP': <atoti.measure.Measure object at 0x120445f90>, 'price.SUM': <atoti.measure.Measure object at 0x120447940>, 'price.MEAN': <atoti.measure.Measure object at 0x120444550>, 'mileage.SUM': <atoti.measure.Measure object at 0x120460850>, 'mileage.MEAN': <atoti.measure.Measure object at 0x120461270>}

In [50]:
# session.close()